In [2]:


"""
使用@jit装饰器对函数进行编译，使得函数的运行速度得到提升。使用@jit(nopython=True)装饰器，可以使得编译后的函数在运行时不依赖于Python解释器，从而提升运行速度。
使用@njit装饰器，可以使得编译后的函数在运行时不依赖于Python解释器，并且使用纯C语言编写，从而提升运行速度
"""

'\n使用@jit装饰器对函数进行编译，使得函数的运行速度得到提升。使用@jit(nopython=True)装饰器，可以使得编译后的函数在运行时不依赖于Python解释器，从而提升运行速度。\n使用@njit装饰器，可以使得编译后的函数在运行时不依赖于Python解释器，并且使用纯C语言编写，从而提升运行速度\n'

In [1]:
"""
大循环、纯数值、numpy 数组 → 用 Numba，起飞
字符串、字典列表、Pandas/Polars → 别用 Numba
超短小函数 → 别加 JIT，浪费编译时间
一律优先用 @njit，报错就说明不适合 JIT
"""

'\n大循环、纯数值、numpy 数组 → 用 Numba，起飞\n字符串、字典列表、Pandas/Polars → 别用 Numba\n超短小函数 → 别加 JIT，浪费编译时间\n一律优先用 @njit，报错就说明不适合 JIT\n'

In [59]:
import numpy as np
from numba import jit


@jit(nopython=True)
def max_jit(x, y):
    res = np.empty_like(x)
    for i in range(len(x)):
        res[i] = max(x[i], y[i])
    return res

In [ ]:
x = np.random.rand(100000)
y = np.random.rand(100000)
#  jit函数调用方式
%timeit max_jit(x,y)
#  numpy函数调用方式
%timeit np.maximum(x,y)
# 原有函数调用方式
%timeit max_jit.py_func(x,y)

57 μs ± 3.63 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
70.4 μs ± 7.79 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
36.5 ms ± 1.14 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [61]:
from numba import jit
import numpy as np


# parallel=True ，在编译时开启并行模式，并行化代码，提高运行效率。在这种情况下可以将for循环中range替换为prange，并行化循环。
@jit(nopython=True, parallel=True)
def max_jit_parallel(x, y):
    res = np.empty_like(x)
    for i in prange(len(x)):
        res[i] = max(x[i], y[i])
    return res

In [ ]:
x = np.random.rand(100000)
y = np.random.rand(100000)
#  jit函数调用方式
%timeit max_jit_parallel(x,y)
#  numpy函数调用方式
%timeit np.maximum(x,y)
#  原有函数调用方式  
%timeit max_jit_parallel.py_func(x,y)

55 μs ± 4.57 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
94.1 μs ± 19.7 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
41.4 ms ± 4.97 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
from numba import njit, prange
import numpy as np


# parallel=True ，在编译时开启并行模式，并行化代码，提高运行效率。在这种情况下可以将for循环中range替换为prange，并行化循环。
@njit(parallel=True)
def max_njit_parallel(x, y):
    res = np.empty_like(x)
    for i in prange(len(x)):
        res[i] = max(x[i], y[i])
    return res

In [67]:
x = np.random.rand(100000)
y = np.random.rand(100000)
#  njit函数调用方式
%timeit max_njit_parallel(x,y)
#  numpy函数调用方式
%timeit np.maximum(x,y)
#  纯python函数调用方式
%timeit max_njit_parallel.py_func(x,y)

50.1 μs ± 2.72 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
71 μs ± 7.86 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
39.7 ms ± 3.84 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [56]:
# 使用vectorize函数加速numpy的计算速度
import numpy as np
from numba import vectorize


@vectorize(nopython=True)
def add_func1(a, b):
    return a + b


a = np.random.rand(1000000)
b = np.random.rand(1000000)

%timeit add_func1(a, b)


4.67 ms ± 317 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [8]:
# 使用vectorize函数加速numpy的计算速度,如果指定使用target='parallel'则可以自动使用多线程加速,必须明确返回值的类型，否则会报错
import numpy as np
from numba import vectorize


@vectorize(['f8(f8, f8)', 'f8(f8, f8)'], nopython=True, target='parallel', )
def add_func2(a, b):
    return a + b


a = np.random.rand(1000000)
b = np.random.rand(1000000)

%timeit add_func2(a, b)

6.36 ms ± 164 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
